### Segment-Wise ISC Analysis

In [1]:
import numpy as np
import pandas as pd
import os, glob
from pathlib import Path

try:
    import utils.process_analysis
    import utils.segment_isc_utils 

except ImportError as e:
    print(f"\nSTILL FAILING: {e}")

### Set up Segments and Data

In [2]:
DATA_DIR        = "../data/02_processed_combined_data"
CONDITIONS_FILE = "../data/sub_conditions.csv"
OUTPUT_DIR      = "../data/04_analysis/h2_segment_results/all_subjects/"

if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)

SEGMENTS = {    'PC':  {1: (9, 54), 2: (55, 205), 3: (205, 243), 4: (243, 278), 5: (279, 319)},
                'PCM': {1: (9, 54), 2: (55, 205), 3: (205, 240), 4: (240, 278), 5: (279, 319)}  }

signals = ['Gaze_X', 'Gaze_Y', 'Pupil_Mean', 'Heart_Rate', 'GSR']

### Main Loop Analysis

In [3]:
pc_subs  = utils.process_analysis.get_subjects_by_condition(CONDITIONS_FILE, target_condition='PC', exclude_prior_exposure=False)
pcm_subs = utils.process_analysis.get_subjects_by_condition(CONDITIONS_FILE, target_condition='PCM', exclude_prior_exposure=False)

# --- MAIN LOOP ---
master_results = []
print("Starting Segment-Wise Analysis...")

for signal in signals:
    print(f"  Processing {signal}...")
    try:
        # Load Data
        df_pc  = utils.process_analysis.load_physio_data(DATA_DIR, pc_subs, signal_type=signal)
        df_pcm = utils.process_analysis.load_physio_data(DATA_DIR, pcm_subs, signal_type=signal)
        
        # Trim to common length
        L = min(df_pc.shape[0], df_pcm.shape[0])
        df_pc, df_pcm = df_pc.iloc[:L], df_pcm.iloc[:L]
        
        # Extract Segments
        segs_pc  = utils.segment_isc_utils.extract_segments(df_pc, SEGMENTS['PC'], L)
        segs_pcm = utils.segment_isc_utils.extract_segments(df_pcm, SEGMENTS['PCM'], L)

        # 1. COMMON (Segs 1, 2, 5)
        d_com_pc  = pd.concat([segs_pc[1], segs_pc[2], segs_pc[5]], axis=0)
        d_com_pcm = pd.concat([segs_pcm[1], segs_pcm[2], segs_pcm[5]], axis=0)
        
        # 2. SCRAMBLED (Segs 3, 4 - Time Aligned / Content Mismatch)
        d_scr_pc  = pd.concat([segs_pc[3], segs_pc[4]], axis=0)
        d_scr_pcm = pd.concat([segs_pcm[3], segs_pcm[4]], axis=0)
        # Trim scrambled to min length
        min_s = min(d_scr_pc.shape[0], d_scr_pcm.shape[0])
        d_scr_pc, d_scr_pcm = d_scr_pc.iloc[:min_s], d_scr_pcm.iloc[:min_s]
        
        # 3. UNSCRAMBLED (Segs 3, 4 - Content Aligned / Time Mismatch)
        # PC: 3 then 4. PCM: 4 then 3. We reorder PCM to match PC.
        d_un_pc  = pd.concat([segs_pc[3], segs_pc[4]], axis=0)
        d_un_pcm = pd.concat([segs_pcm[4], segs_pcm[3]], axis=0) # SWAPPED
        # Trim unscrambled
        min_u = min(d_un_pc.shape[0], d_un_pcm.shape[0])
        d_un_pc, d_un_pcm = d_un_pc.iloc[:min_u], d_un_pcm.iloc[:min_u]
        
        # Calculate Stats
        tasks = [("Common", d_com_pc, d_com_pcm),
                 ("Scrambled", d_scr_pc, d_scr_pcm),
                 ("Unscrambled", d_un_pc, d_un_pcm)]
        
        for cond, da, db in tasks:
            m, el, eh, p = utils.process_analysis.calculate_permuted_stats(da, db)
            master_results.append({
                "Signal": signal, 
                "Condition": cond, 
                "Mean_r": m, 
                "Err_Low": el, 
                "Err_High": eh,
                "p-value": p
            })
            
    except Exception as e:
        print(f"    Error {signal}: {e}")

# Save
pd.DataFrame(master_results).to_csv(os.path.join(OUTPUT_DIR, "h2_segment_stats.csv"), index=False)
print("Computations Complete.")

Starting Segment-Wise Analysis...
  Processing Gaze_X...
  Processing Gaze_Y...
  Processing Pupil_Mean...
  Processing Heart_Rate...
  Processing GSR...
Computations Complete.


In [5]:
OUTPUT_DIR

'../data/04_analysis/h2_segment_results/all_subjects/'

In [6]:
os.getcwd()

'/Users/schmaelz/Documents/01_GITHUB/nomcomm/GITHUB_PAPERS_WORKING/PC_HJ/scripts'